In [1]:
# pandas for data manipulation
# re for regular expressions
import re

# graph helpers
import numpy as np
import pandas as pd

from utils.graphs import plot_magnitude_analysis_interactive
from utils.import_data import get_csv_files, print_user_location_tables, sort_meta_info

In [2]:
path_to: str = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project"
path: str = path_to + "\\CSI DATA"

# scenario -> location -> user -> esp -> trial -> file_path
FileMap = dict[str, dict[str, dict[str, dict[str, dict[str, str]]]]]
# scenario -> location -> user -> esp -> trial -> csi ndarray
csi_map = dict[str, dict[str, dict[str, dict[str, dict[str, np.ndarray]]]]]
# scenario -> location -> user -> esp -> trial -> agc gain ndarray
agc_gain_map = dict[str, dict[str, dict[str, dict[str, dict[str, np.ndarray]]]]]


all_data_files = get_csv_files(path)
scenarios_id, locations_id, users_id, esps_id, trials_id = sort_meta_info(path)

print(f"Scenarios present: {', '.join(scenarios_id) or 'none'}")
print_user_location_tables(all_data_files)


Scenarios present: 1

User 01 - Pedro Monteiro
    1  2  3  4  5  6  7  8  9 10 11 12 13 14
A | .  .  .  .  .  .  .  .  .  .  .  .  .  .
B | .  .  .  .  .  .  .  .  .  .  .  .  .  .
C | .  .  .  .  .  .  .  .  .  .  .  .  .  .
D | .  .  .  .  .  .  .  .  .  .  .  .  .  .
E | .  .  .  .  .  .  .  .  .  .  .  .  .  .
F | .  .  .  .  .  .  .  .  .  .  .  .  .  .
No files found for this user.

User 02 - Guilherme Cabaço
    1  2  3  4  5  6  7  8  9 10 11 12 13 14
A | X  X  .  .  X  .  .  .  .  .  .  .  X  X
B | X  X  .  .  X  .  .  .  .  X  X  X  X  X
C | X  X  X  X  X  X  X  X  X  X  X  .  .  X
D | X  X  X  X  X  X  X  X  .  .  .  .  .  .
E | X  X  X  X  X  X  X  X  .  X  X  X  X  .
F | .  .  .  .  X  .  .  X  .  X  X  X  X  .
Other locations: Z-0

User 03 - Henrique
    1  2  3  4  5  6  7  8  9 10 11 12 13 14
A | X  X  .  .  X  .  .  .  .  .  .  .  X  X
B | X  X  .  .  X  .  .  X  .  X  X  X  X  X
C | X  X  X  X  X  X  X  X  X  X  X  .  .  X
D | X  X  X  X  X  X  X  X  .  .  .  .  .  .

In [9]:
def process_csi(data_file: str, its5ghz: bool) -> tuple[np.ndarray, np.ndarray, np.ndarray, int, int, int]:
    # 5 GHz saved CSV header from app_main.c UDP payload: type,seq,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
    # 2.4 GHz saved CSV header from app_main.c UDP payload: type,id,mac,rssi,rate,sig_mode,mcs,bandwidth,smoothing,not_sounding,aggregation,stbc,fec_coding,sgi,noise_floor,ampdu_cnt,channel,secondary_channel,local_timestamp,ant,sig_len,rx_state,len,first_word,data
    csi_raw: pd.Series
    agc_raw: pd.Series | None = None
    valid_agc_gains: list[int] = []

    # number of samples
    if its5ghz:
        # print("\tCom 5 GHz")
        file_csv = pd.read_csv(data_file, header=None, usecols=[7, 14])
        csi_raw: pd.Series = file_csv.iloc[:, 1]
        agc_raw = file_csv.iloc[:, 0]
    else:
        file_csv = pd.read_csv(data_file, header=None, usecols=[24])
        csi_raw: pd.Series = file_csv.iloc[:, 0]

    total_values_2_4: int = 128
    total_sc_5: int = 114
    valid_csi: list[list[float]] = []

    no_match_count: int = 0
    no_complete_count: int = 0

    for row_index, entry in csi_raw.items():
        match = re.search(r"\[(.*?)\]", str(entry))
        if not match:
            no_match_count += 1
            continue

        nums = [float(n) for n in re.findall(r"-?\d+", match.group(1))]

        if (its5ghz and len(nums) == total_sc_5) or (not its5ghz and len(nums) == total_values_2_4):
            valid_csi.append(nums)
            if agc_raw is not None:
                valid_agc_gains.append(int(agc_raw.loc[row_index]))
        else:
            no_complete_count += 1

    valid_csi = np.array(valid_csi)
    agc_gains = np.array(valid_agc_gains, dtype=int)

    # print(f"\tTotal CSI entries: {len(csi_raw)}")
    # print(f"\tValid CSI entries: {len(valid_csi)}")
    # print(f"\tInvalid CSI entries (no match): {no_match_count}")
    # print(f"\tInvalid CSI entries (incomplete): {no_complete_count}")
    # print(f"\tValid CSI shape: {valid_csi.shape}\n")

    if not its5ghz:
        # (n_amostras, n_subcarriers)
        real = valid_csi[:, 1::2]
        imag = valid_csi[:, ::2]
        complex_csi = real + 1j * imag

        # coloca sc DC no centro (index 32)
        fft_csi = np.fft.fftshift(complex_csi, axes=1)

        # Extract the 52 active subcarriers (IEEE 802.11n standard): [6:58]
        # # remove guards
        # # (n_amostras, 52)
        active_sc = fft_csi[:, 6:58]

        # Remove subcarriers at position DC = 26 (center) e 27 (0)
        # (n_amostras, 51)
        active_sc = np.delete(active_sc, [26, 27], axis=1)
        mag = np.abs(active_sc)
        #phase = np.angle(active_sc)
    else:
        imag = valid_csi[:, ::2]
        real = valid_csi[:, 1::2]
        complex_csi = real + 1j * imag
        mag = np.abs(complex_csi)

        # Remove subcarrier of 0 magnitude
        mag = np.delete(mag, [28], axis=1)
        #phase = np.angle(complex_csi)

    return mag, agc_gains, no_match_count, no_complete_count, len(csi_raw)


def process_files(data_files: FileMap) -> tuple[csi_map, csi_map, agc_gain_map]:
    magnitudes = {}
    agc_gain_data = {}
    no_match_count: int = 0
    no_complete_count: int = 0
    total_entries: int = 0

    for scenario_key, locations_map in data_files.items():
        # print(f"Processing scenario: {scenario_key}")
        magnitudes[scenario_key] = {}
        agc_gain_data[scenario_key] = {}

        for location_key, users_map in locations_map.items():
            magnitudes[scenario_key][location_key] = {}
            agc_gain_data[scenario_key][location_key] = {}

            for user_key, esps_map in users_map.items():
                magnitudes[scenario_key][location_key][user_key] = {}
                agc_gain_data[scenario_key][location_key][user_key] = {}

                for esp_key, trials_map in esps_map.items():
                    magnitudes[scenario_key][location_key][user_key][esp_key] = {}
                    esp_id = int(esp_key.removeprefix("esp_"))
                    its5ghz = 11 <= esp_id <= 20
                    #print("Esp", esp_key, "5GHz:", its5ghz)

                    for trial_key, file_path in trials_map.items():
                        if file_path is None:
                            continue

                        (
                            magnitudes[scenario_key][location_key][user_key][esp_key][trial_key],
                            agc_gains,
                            no_match,
                            no_complete,
                            total,
                        ) = process_csi(str(file_path), its5ghz)

                        if its5ghz:
                            agc_gain_data[scenario_key][location_key][user_key].setdefault(esp_key, {})
                            agc_gain_data[scenario_key][location_key][user_key][esp_key][trial_key] = agc_gains
                            unique_agc_gains = sorted(set(agc_gains.tolist()))
                            agc_gain_text = ", ".join(f"{gain:g}" for gain in unique_agc_gains) or "none"
                            #print(f"\tAGC {location_key}/{user_key}/{esp_key}/{trial_key}: n={len(agc_gains)}, values={agc_gain_text}")

                        no_match_count += no_match
                        no_complete_count += no_complete
                        total_entries += total

        # print(f"Total invalid CSI entries (no match): {no_match_count}")
        # print(f"Total invalid CSI entries (incomplete): {no_complete_count}")
        # if total_entries > 0:
        #     print(f"Total percentage of no match: {no_match_count / total_entries:.2%}")
        #     print(f"Total percentage of incomplete: {no_complete_count / total_entries:.2%}\n\n")
        # else:
        #     print("No entries processed.\n\n")

        no_match_count = 0
        no_complete_count = 0
        total_entries = 0

    return magnitudes, agc_gain_data

In [10]:
magnitude_data, agc_gain_data = process_files(all_data_files)

In [ ]:
# Asks what graph subset to draw before plotting.
# Increase packet_stride_3d if the 3D plots are slow to render.
plot_magnitude_analysis_interactive(magnitude_data)

### Pre Processing

In [ ]:
def segment_signal(
    signal: np.ndarray,
    window_size: int,
    overlap: int,
) -> list[np.ndarray]:
    if overlap >= window_size:
        raise ValueError("Overlap must be less than window size")

    step = window_size - overlap
    segments: list[np.ndarray] = []

    for i in range(0, signal.shape[0] - window_size + 1, step):
        segments.append(signal[i : i + window_size])

    return segments


def calibrate_magnitude(magnitude: np.ndarray) -> np.ndarray:
    # Calibrate magnitude by subtracting the mean of each subcarrier across all samples
    mean_per_subcarrier = np.mean(magnitude, axis=0)
    calibrated_magnitude = magnitude - mean_per_subcarrier
    return calibrated_magnitude


def process_pipeline_raw_magnitude(
    csi_magnitude: np.ndarray,
    window_size: int = 60,
    overlap_size: int = 30,
) -> np.ndarray:
    csi_magnitude = calibrate_magnitude(csi_magnitude)

    segmented_data = segment_signal(csi_magnitude, window_size, overlap_size)

    # Extract features per window
    features = []
    for window_data in segmented_data:
        # Flatten window to compute scalar statistics
        flat_window = window_data.flatten()
        features.append(
            [
                np.mean(flat_window),
                np.std(flat_window),
                np.max(flat_window),
                np.var(flat_window),
                np.sum(flat_window**2),
            ]
        )

    return np.array(features)  # shape: (n_windows, 5)